# 01 — Data Exploration

Quick-look visualisations for each raw dataset so the whole team understands what we're working with.

**Run order:** Run all cells top-to-bottom after `python scripts/download_data.py` has completed.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path

RAW   = Path('../data/raw')
PROC  = Path('../data/processed')

print('Libraries loaded.')

---
## 1. Master Dataset — Shape & Completeness

In [ ]:
master = pd.read_parquet(PROC / 'master.parquet')
print(f'Shape: {master.shape}')
master.head()

In [ ]:
# Null % per column — quickly shows which joins are incomplete
null_pct = (master.isnull().mean() * 100).round(1).reset_index()
null_pct.columns = ['column', 'null_%']

fig = px.bar(
    null_pct, x='column', y='null_%',
    title='Master Dataset — % Null Values per Column',
    labels={'null_%': '% Null', 'column': ''},
    color='null_%', color_continuous_scale='Reds',
    text='null_%'
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(showlegend=False, yaxis_range=[0, 105])
fig.show()

In [ ]:
# Row counts by year — see where most data lives
year_counts = master.groupby('year').size().reset_index(name='rows')

fig = px.bar(
    year_counts, x='year', y='rows',
    title='Master Dataset — Rows per Year',
    labels={'rows': 'Row Count', 'year': 'Year'}
)
fig.show()

---
## 2. HNO 2026 — People in Need by Country & Sector

In [ ]:
hno = pd.read_csv(RAW / 'hno_2026.csv')
hno.columns = hno.columns.str.strip().str.lower().str.replace(' ', '_')
print(f'Shape: {hno.shape}  |  Columns: {hno.columns.tolist()}')
hno.head()

In [ ]:
# Country-level total PIN (cluster == ALL)
country_pin = (
    hno[hno['cluster'] == 'ALL']
    .rename(columns={'country_iso3': 'iso3', 'in_need': 'pin'})
    [['iso3', 'pin', 'population']]
    .dropna(subset=['pin'])
    .sort_values('pin', ascending=False)
)
country_pin['pin_pct'] = (country_pin['pin'] / country_pin['population'] * 100).round(1)

fig = px.bar(
    country_pin, x='iso3', y='pin',
    title='HNO 2026 — People in Need by Country (total caseload)',
    labels={'pin': 'People in Need', 'iso3': 'Country'},
    color='pin_pct', color_continuous_scale='OrRd',
    color_continuous_midpoint=country_pin['pin_pct'].median(),
    hover_data=['pin_pct', 'population'],
    text_auto='.2s'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Sector breakdown — stacked bar across all countries
sector_data = (
    hno[hno['cluster'] != 'ALL']
    .rename(columns={'country_iso3': 'iso3', 'in_need': 'pin', 'cluster': 'sector'})
    .dropna(subset=['pin'])
)

# Keep only top-level sectors (no sub-sectors like PRO-GBV)
top_sectors = ['FSC', 'HEA', 'EDU', 'NUT', 'PRO', 'SHL', 'WSH', 'MPC']
sector_data = sector_data[sector_data['sector'].isin(top_sectors)]

fig = px.bar(
    sector_data.sort_values('pin', ascending=False),
    x='iso3', y='pin', color='sector', barmode='stack',
    title='HNO 2026 — People in Need by Country and Sector',
    labels={'pin': 'People in Need', 'iso3': 'Country', 'sector': 'Sector'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

---
## 3. FTS Funding — Coverage Ratio Distribution

In [ ]:
fts = pd.read_csv(RAW / 'funding.csv')
fts.columns = fts.columns.str.lower()
fts = fts.rename(columns={'countrycode': 'iso3', 'requirements': 'req_usd',
                           'funding': 'fund_usd', 'percentfunded': 'pct_funded'})

# Filter to rows with real plans and recent years
fts_clean = fts[(fts['year'] >= 2015) & fts['req_usd'].notna() & (fts['req_usd'] > 0)].copy()
fts_clean['coverage'] = (fts_clean['fund_usd'] / fts_clean['req_usd']).clip(0, 1.5)

print(f'FTS rows with requirements > 0 (2015+): {len(fts_clean):,}')
fts_clean.head()

In [ ]:
fig = px.histogram(
    fts_clean, x='coverage', nbins=40,
    title='FTS — Distribution of Funding Coverage Ratios (all plans 2015–2026)',
    labels={'coverage': 'Coverage Ratio (funding / requirements)'},
    color_discrete_sequence=['#e63946']
)
fig.add_vline(x=1.0, line_dash='dash', line_color='green', annotation_text='Fully funded')
fig.add_vline(x=fts_clean['coverage'].median(), line_dash='dot', line_color='orange',
              annotation_text=f'Median {fts_clean["coverage"].median():.0%}')
fig.show()

In [ ]:
# Coverage trend over years — median coverage per year
yearly = fts_clean.groupby('year')['coverage'].median().reset_index()

fig = px.line(
    yearly, x='year', y='coverage',
    title='FTS — Median Funding Coverage Ratio by Year',
    labels={'coverage': 'Median Coverage Ratio', 'year': 'Year'},
    markers=True
)
fig.add_hline(y=1.0, line_dash='dash', line_color='green', annotation_text='Fully funded')
fig.update_layout(yaxis_tickformat='.0%')
fig.show()

In [ ]:
# Scatter: requirements vs funding (log scale) — each dot is one plan
fts_2025 = fts_clean[fts_clean['year'] == 2025].copy()

fig = px.scatter(
    fts_2025, x='req_usd', y='fund_usd',
    hover_name='name', color='coverage',
    color_continuous_scale='RdYlGn',
    log_x=True, log_y=True,
    title='FTS 2025 — Requirements vs Funding per Plan (log scale)',
    labels={'req_usd': 'Requirements (USD)', 'fund_usd': 'Funding Received (USD)'}
)
# Perfect-funding diagonal
line_range = [1e6, 5e9]
fig.add_trace(go.Scatter(
    x=line_range, y=line_range,
    mode='lines', line=dict(dash='dash', color='grey'),
    name='100% funded'
))
fig.show()

---
## 4. World Map — Funding Coverage by Country (2025)

In [ ]:
# Aggregate FTS 2025 to country level
cov_2025 = (
    fts_clean[fts_clean['year'] == 2025]
    .groupby('iso3').agg(req=('req_usd', 'sum'), fund=('fund_usd', 'sum'))
    .reset_index()
)
cov_2025['coverage'] = (cov_2025['fund'] / cov_2025['req']).clip(0, 1)
cov_2025['gap'] = 1 - cov_2025['coverage']

fig = px.choropleth(
    cov_2025,
    locations='iso3',
    color='coverage',
    hover_name='iso3',
    hover_data={'req': ':,.0f', 'fund': ':,.0f', 'coverage': ':.1%'},
    color_continuous_scale='RdYlGn',
    range_color=[0, 1],
    title='Funding Coverage by Country — 2025  (green = well-funded, red = underfunded)',
    labels={'coverage': 'Coverage Ratio'}
)
fig.update_layout(geo=dict(showframe=False, showcoastlines=True))
fig.show()

In [ ]:
# Same data on a 3-D globe — rotate to any region of interest
fig = px.choropleth(
    cov_2025,
    locations='iso3',
    color='gap',
    hover_name='iso3',
    hover_data={'req': ':,.0f', 'fund': ':,.0f', 'coverage': ':.1%'},
    color_continuous_scale='Reds',
    range_color=[0, 1],
    title='Funding Gap on the Globe — 2025  (darker red = larger gap)',
    labels={'gap': 'Funding Gap'}
)
fig.update_geos(
    projection_type='orthographic',
    showland=True, landcolor='lightgray',
    showocean=True, oceancolor='lightblue',
    showcoastlines=True, coastlinecolor='white',
    showframe=False
)
fig.update_layout(height=600)
fig.show()

---
## 5. HNO — People in Need on the World Map (2026)

In [ ]:
# Flat world map
fig = px.choropleth(
    country_pin,
    locations='iso3',
    color='pin',
    hover_name='iso3',
    hover_data={'pin': ':,.0f', 'pin_pct': True},
    color_continuous_scale='YlOrRd',
    title='People in Need by Country — HNO 2026',
    labels={'pin': 'People in Need', 'pin_pct': '% of Population'}
)
fig.update_layout(geo=dict(showframe=False, showcoastlines=True))
fig.show()

In [ ]:
# Globe view of people in need
fig = px.choropleth(
    country_pin,
    locations='iso3',
    color='pin',
    hover_name='iso3',
    color_continuous_scale='YlOrRd',
    title='People in Need — Globe View (HNO 2026)',
    labels={'pin': 'People in Need'}
)
fig.update_geos(
    projection_type='orthographic',
    showland=True, landcolor='lightgray',
    showocean=True, oceancolor='#a8d5e2',
    showcoastlines=True, coastlinecolor='white',
    showframe=False
)
fig.update_layout(height=600)
fig.show()

---
## 6. Combined — Need vs Coverage Bubble Map (2025)

In [ ]:
# Merge HNO PIN with FTS coverage for a combined picture
pin_map = country_pin.rename(columns={'iso3': 'country_iso3'})
cov_map = cov_2025.rename(columns={'iso3': 'country_iso3'})
combined = pin_map.merge(cov_map, on='country_iso3', how='inner')
combined['gap_score'] = (1 - combined['coverage']) * np.log1p(combined['pin'])

print(f'{len(combined)} countries with both PIN and funding data')

fig = px.scatter(
    combined.sort_values('gap_score', ascending=False),
    x='coverage', y='pin',
    size='gap_score', color='gap_score',
    hover_name='country_iso3',
    log_y=True,
    color_continuous_scale='Reds',
    title='Need vs Coverage — Bubble size = Gap Score  (top-right = most overlooked)',
    labels={'coverage': 'Funding Coverage Ratio', 'pin': 'People in Need (log scale)'}
)
fig.add_vline(x=0.5, line_dash='dash', line_color='grey', annotation_text='50% coverage')
fig.update_layout(yaxis_tickformat='.2s')
fig.show()

---
## 7. geopandas Static Map (matplotlib)

In [ ]:
# Load Natural Earth world boundaries bundled with geopandas
# Natural Earth shapefile — cached to data/raw/ne_world.gpkg by scripts/download_data.py
_ne_path = Path('../data/raw/ne_world.gpkg')
if not _ne_path.exists():
    import geopandas as gpd
    world = gpd.read_file('https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip')
    world.to_file(_ne_path, driver='GPKG')
world = gpd.read_file(_ne_path)

# Merge coverage data
world_cov = world.merge(
    cov_2025.rename(columns={'iso3': 'iso_a3'}),
    on='iso_a3', how='left'
)

fig, ax = plt.subplots(1, 1, figsize=(18, 9))
world.plot(ax=ax, color='#d3d3d3', edgecolor='white', linewidth=0.4)
world_cov.dropna(subset=['coverage']).plot(
    ax=ax, column='coverage',
    cmap='RdYlGn', vmin=0, vmax=1,
    edgecolor='white', linewidth=0.4,
    legend=True,
    legend_kwds={'label': 'Funding Coverage Ratio', 'shrink': 0.5}
)
ax.set_title('Humanitarian Funding Coverage by Country — 2025', fontsize=14, pad=12)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Orthographic globe projection via geopandas + matplotlib
import matplotlib.patches as mpatches

try:
    world_cov_proj = world_cov.to_crs('+proj=ortho +lat_0=10 +lon_0=25')
    world_proj     = world.to_crs('+proj=ortho +lat_0=10 +lon_0=25')

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    world_proj.plot(ax=ax, color='#d3d3d3', edgecolor='white', linewidth=0.3)
    world_cov_proj.dropna(subset=['coverage']).plot(
        ax=ax, column='coverage',
        cmap='RdYlGn', vmin=0, vmax=1,
        edgecolor='white', linewidth=0.3,
        legend=True,
        legend_kwds={'label': 'Funding Coverage', 'shrink': 0.5, 'orientation': 'horizontal'}
    )
    ax.set_title('Humanitarian Funding Coverage — Globe View (Africa/MENA centred)', fontsize=13, pad=12)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Globe projection failed: {e}')
    print('Install pyproj >= 3.x and try again, or use the Plotly globe above.')

---
## 8. CBPF — Pooled Fund Allocations by Country & Year

In [ ]:
import json
with open(RAW / 'cbpf.json') as f:
    cbpf_raw = json.load(f)
cbpf = pd.DataFrame(cbpf_raw['value'])
cbpf.columns = cbpf.columns.str.lower()
cbpf = cbpf.rename(columns={'pooledfundname': 'country', 'allocationyear': 'year',
                              'approvedbudget': 'approved_usd'})
cbpf['approved_usd'] = pd.to_numeric(cbpf['approved_usd'], errors='coerce')
print(f'CBPF rows: {len(cbpf)} | Years: {cbpf["year"].min()}–{cbpf["year"].max()}')
cbpf.head()

In [ ]:
# Total CBPF allocations by country (all years)
cbpf_total = cbpf.groupby('country')['approved_usd'].sum().reset_index().sort_values('approved_usd', ascending=False).head(20)

fig = px.bar(
    cbpf_total, x='country', y='approved_usd',
    title='CBPF — Total Pooled Fund Allocations by Country (all years)',
    labels={'approved_usd': 'Total Approved (USD)', 'country': ''},
    color='approved_usd', color_continuous_scale='Blues',
    text_auto='.2s'
)
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
fig.show()

In [ ]:
# CBPF trends over time for top 5 countries
top5 = cbpf_total['country'].head(5).tolist()
cbpf_trend = cbpf[cbpf['country'].isin(top5)].groupby(['year','country'])['approved_usd'].sum().reset_index()

fig = px.line(
    cbpf_trend, x='year', y='approved_usd', color='country',
    title='CBPF — Pooled Fund Allocations Over Time (top 5 countries)',
    labels={'approved_usd': 'Approved Budget (USD)', 'year': 'Year'},
    markers=True
)
fig.update_layout(yaxis_tickformat='.2s')
fig.show()